In [ ]:
"""
CS3807 - Deep Learning Laboratory
Experiment 4: Comparative Study of Deep CNN Architectures Using Transfer Learning
Dataset: CIFAR-10

This script performs:
  Task 1: Dataset preparation
  Task 2: Transfer learning setup (VGG16 / ResNet50 / MobileNetV2)
  Task 3: Model training (feature extraction, frozen base)
  Task 4: Fine tuning (unfreeze last block)
  Task 5: Model evaluation (accuracy, precision, recall, F1, confusion matrix)
  Bonus: Hyperparameter study and simple architecture comparison table
"""

import os
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import seaborn as sns

# -----------------------------------------------------------------------
# Reproducibility
# -----------------------------------------------------------------------
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================================
# TASK 1: DATASET PREPARATION
# =========================================================================
def load_and_prepare_data():
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

    # Normalize pixel values to [0, 1]
    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    y_train_cat = to_categorical(y_train, num_classes=10)
    y_test_cat = to_categorical(y_test, num_classes=10)

    print("Training data shape :", x_train.shape)
    print("Testing data shape  :", x_test.shape)
    print("Training labels     :", y_train_cat.shape)
    print("Testing labels      :", y_test_cat.shape)

    # Display 10 sample images
    plt.figure(figsize=(10, 4))
    for i in range(10):
        plt.subplot(2, 5, i + 1)
        plt.imshow(x_train[i])
        plt.title(CLASS_NAMES[int(y_train[i])])
        plt.axis("off")
    plt.suptitle("Sample CIFAR-10 Images")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/sample_images.png", dpi=150)
    plt.close()

    return (x_train, y_train, y_train_cat), (x_test, y_test, y_test_cat)


# =========================================================================
# TASK 2: TRANSFER LEARNING - BUILD MODEL
# =========================================================================
def build_transfer_model(base_name="VGG16", input_shape=(96, 96, 3), num_classes=10):
    """
    Loads a pretrained CNN, freezes the convolutional base, and attaches
    a new classification head.
    """
    if base_name == "VGG16":
        base_model = VGG16(weights="imagenet", include_top=False, input_shape=input_shape)
    elif base_name == "ResNet50":
        base_model = ResNet50(weights="imagenet", include_top=False, input_shape=input_shape)
    elif base_name == "MobileNetV2":
        base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=input_shape)
    else:
        raise ValueError("Unsupported base model")

    # Freeze convolutional base
    base_model.trainable = False

    inputs = layers.Input(shape=(32, 32, 3))
    x = layers.Resizing(input_shape[0], input_shape[1])(inputs)   # upscale CIFAR images
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    return model, base_model


# =========================================================================
# TASK 3: MODEL TRAINING (frozen base / feature extraction)
# =========================================================================
def train_model(model, x_train, y_train, x_val, y_val,
                 lr=0.001, batch_size=32, epochs=10, optimizer_name="Adam"):
    opt = optimizers.Adam(learning_rate=lr) if optimizer_name == "Adam" \
        else optimizers.SGD(learning_rate=lr, momentum=0.9)

    model.compile(optimizer=opt, loss="categorical_crossentropy", metrics=["accuracy"])

    start = time.time()
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        batch_size=batch_size,
        epochs=epochs,
        verbose=2
    )
    train_time = time.time() - start
    return history, train_time


# =========================================================================
# TASK 4: FINE TUNING
# =========================================================================
def fine_tune_model(model, base_model, x_train, y_train, x_val, y_val,
                     lr=1e-5, batch_size=32, epochs=5, unfreeze_last_n=4):
    base_model.trainable = True
    # Freeze all layers except the last `unfreeze_last_n`
    for layer in base_model.layers[:-unfreeze_last_n]:
        layer.trainable = False

    model.compile(optimizer=optimizers.Adam(learning_rate=lr),
                   loss="categorical_crossentropy", metrics=["accuracy"])

    start = time.time()
    history_ft = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        batch_size=batch_size,
        epochs=epochs,
        verbose=2
    )
    ft_time = time.time() - start
    return history_ft, ft_time


# =========================================================================
# TASK 5: MODEL EVALUATION
# =========================================================================
def evaluate_model(model, x_test, y_test_cat, y_test_labels):
    y_pred_probs = model.predict(x_test)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = y_test_labels.flatten()

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro")
    rec = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES)

    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}")
    print(report)

    # Confusion matrix plot
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png", dpi=150)
    plt.close()

    # Misclassified images (optional)
    misclassified_idx = np.where(y_pred != y_true)[0][:10]
    if len(misclassified_idx) > 0:
        plt.figure(figsize=(10, 4))
        for i, idx in enumerate(misclassified_idx):
            plt.subplot(2, 5, i + 1)
            plt.imshow(x_test[idx])
            plt.title(f"T:{CLASS_NAMES[y_true[idx]]}\nP:{CLASS_NAMES[y_pred[idx]]}", fontsize=8)
            plt.axis("off")
        plt.suptitle("Misclassified Images")
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/misclassified_images.png", dpi=150)
        plt.close()

    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1,
            "confusion_matrix": cm, "report": report}


def plot_training_curves(history, history_ft=None, tag=""):
    acc = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]

    if history_ft is not None:
        acc += history_ft.history["accuracy"]
        val_acc += history_ft.history["val_accuracy"]
        loss += history_ft.history["loss"]
        val_loss += history_ft.history["val_loss"]

    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(6, 4))
    plt.plot(epochs_range, acc, label="Training Accuracy")
    plt.plot(epochs_range, val_acc, label="Validation Accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()
    plt.title(f"Accuracy Curve {tag}")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/accuracy_curve_{tag}.png", dpi=150)
    plt.close()

    plt.figure(figsize=(6, 4))
    plt.plot(epochs_range, loss, label="Training Loss")
    plt.plot(epochs_range, val_loss, label="Validation Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
    plt.title(f"Loss Curve {tag}")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/loss_curve_{tag}.png", dpi=150)
    plt.close()


# =========================================================================
# BONUS: SIMPLE HYPERPARAMETER STUDY
# =========================================================================
def hyperparameter_study(x_train, y_train, x_val, y_val, x_test, y_test_cat, y_test_labels):
    """
    Runs a small grid over learning rate and batch size (kept small so the
    study finishes quickly -- extend for the full report as required).
    """
    results = []
    configs = [
        {"lr": 0.001, "batch_size": 32, "epochs": 5, "optimizer_name": "Adam"},
        {"lr": 0.0001, "batch_size": 32, "epochs": 5, "optimizer_name": "Adam"},
        {"lr": 0.001, "batch_size": 64, "epochs": 5, "optimizer_name": "SGD"},
    ]

    for cfg in configs:
        model, base_model = build_transfer_model("VGG16")
        _, train_time = train_model(model, x_train, y_train, x_val, y_val, **cfg)
        metrics = evaluate_model(model, x_test, y_test_cat, y_test_labels)
        results.append({**cfg, "accuracy": metrics["accuracy"], "time": train_time})

    return results


# =========================================================================
# MAIN
# =========================================================================
if __name__ == "__main__":
    # ---- Task 1 ----
    (x_train, y_train, y_train_cat), (x_test, y_test, y_test_cat) = load_and_prepare_data()

    # Small validation split from training data
    val_split = int(0.9 * len(x_train))
    x_tr, x_val = x_train[:val_split], x_train[val_split:]
    y_tr, y_val = y_train_cat[:val_split], y_train_cat[val_split:]

    # ---- Task 2 & 3: build + train with frozen base ----
    model, base_model = build_transfer_model(base_name="VGG16")
    history, train_time = train_model(
        model, x_tr, y_tr, x_val, y_val,
        lr=0.001, batch_size=32, epochs=10, optimizer_name="Adam"
    )
    print(f"Feature-extraction training time: {train_time:.2f} sec")

    # Evaluate before fine-tuning
    print("\n--- Evaluation BEFORE fine-tuning ---")
    metrics_before = evaluate_model(model, x_test, y_test_cat, y_test)

    # ---- Task 4: fine-tune ----
    history_ft, ft_time = fine_tune_model(
        model, base_model, x_tr, y_tr, x_val, y_val,
        lr=1e-5, batch_size=32, epochs=5, unfreeze_last_n=4
    )
    print(f"Fine-tuning time: {ft_time:.2f} sec")

    plot_training_curves(history, history_ft, tag="VGG16")

    # ---- Task 5: evaluate after fine-tuning ----
    print("\n--- Evaluation AFTER fine-tuning ---")
    metrics_after = evaluate_model(model, x_test, y_test_cat, y_test)

    print("\nSummary")
    print(f"Accuracy before fine-tuning: {metrics_before['accuracy']:.4f}")
    print(f"Accuracy after  fine-tuning: {metrics_after['accuracy']:.4f}")
    print(f"Total parameters: {model.count_params():,}")

    # ---- Optional: hyperparameter study ----
    # results = hyperparameter_study(x_tr, y_tr, x_val, y_val, x_test, y_test_cat, y_test)
    # for r in results:
    #     print(r)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1477s 9us/step
Training data shape : (50000, 32, 32, 3)
Testing data shape  : (10000, 32, 32, 3)
Training labels     : (50000, 10)
Testing labels      : (10000, 10)


/tmp/ipykernel_619/2428551219.py:65: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  plt.title(CLASS_NAMES[int(y_train[i])])


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
1407/1407 - 71s - 51ms/step - accuracy: 0.5615 - loss: 1.2636 - val_accuracy: 0.6510 - val_loss: 1.0288
Epoch 2/10
1407/1407 - 67s - 48ms/step - accuracy: 0.6443 - loss: 1.0201 - val_accuracy: 0.6786 - val_loss: 0.9492
Epoch 3/10
1407/1407 - 70s - 50ms/step - accuracy: 0.6672 - loss: 0.9567 - val_accuracy: 0.6918 - val_loss: 0.9163
Epoch 4/10
1407/1407 - 70s - 50ms/step - accuracy: 0.6808 - loss: 0.9202 - val_accuracy: 0.6976 - val_loss: 0.8926
Epoch 5/10
1407/1407 - 70s - 50ms/step - accuracy: 0.6885 - loss: 0.8905 - val_accuracy: 0.6976 - val_loss: 0.8789
Epoch 6/10
1407/1407 - 70s - 50ms/step - accuracy: 0.6976 - loss: 0.8674 - val_accuracy: 0.7024 - val_loss: 0.8734
Epoch 7/10
1407/1407 - 70s - 50ms/step - accuracy: 0.7042 - loss: 0.8453 - val_accuracy: 0.7038 - val_loss: 0.8626
Epoch 8/10
1407/1407 - 69s - 49ms/step - accuracy: 0.7096 - loss: 0.8301 - val_accuracy: 0.7066 - val_loss: 0.8599
Epoch 9/10
1407/1407 - 69s - 

In [ ]:
import shutil

# Create a zip archive of the 'outputs' directory
shutil.make_archive('output_images', 'zip', OUTPUT_DIR)
print(f"Successfully created output_images.zip in the current directory. You can download it from the file browser.")

Successfully created output_images.zip in the current directory. You can download it from the file browser.
